In [5]:
!pip install pandas numpy scikit-learn xgboost openpyxl joblib



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [26]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load Dataset
# Ensure "health_nutrition_disease_dataset_12000.xlsx" is in the same folder
file_path = "health_nutrition_disease_dataset_12000.xlsx"
try:
    df = pd.read_excel(file_path)
    print(f"Dataset loaded: {df.shape}")
except FileNotFoundError:
    print("Error: The Excel file was not found. Please check the file path.")

# 2. Data Preprocessing
# Encoding Gender and Disease Risks
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

disease_columns = [
    "Diabetes_Risk", "Hypertension_Risk", "Heart_Disease_Risk",
    "Obesity_Risk", "Anemia_Risk", "Kidney_Disease_Risk"
]

for col in disease_columns:
    df[col] = df[col].map({"High": 1, "Low": 0})

# Define Features
features = [
    "Age", "Gender", "BMI", "Daily_Calories_kcal", "Carbohydrates_g",
    "Protein_g", "Total_Fat_g", "Saturated_Fat_g", "Trans_Fat_g",
    "Total_Sugar_g", "Added_Sugar_g", "Fiber_g", "Sodium_mg",
    "Potassium_mg", "Calcium_mg", "Iron_mg", "Vitamin_D_IU",
    "Vitamin_B12_mcg", "Physical_Activity_min", "Water_Intake_L"
]

X = df[features]

# 3. Scaling & Train-Test Split
X_train_raw, X_test_raw = train_test_split(X, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Save the scaler for future use
joblib.dump(scaler, "scaler.pkl")

# 4. Training Multi-Output Models
models = {}
print("\n--- Training Results ---")

from sklearn.metrics import classification_report, accuracy_score

models = {}

for disease in disease_columns:
    y = df[disease]
    y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)

    # 1. Define the model (using the anti-overfitting parameters we discussed)
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    # 2. Fit the model
    model.fit(X_train_scaled, y_train)

    # 3. Get predictions for both sets to check for overfitting
    train_preds = model.predict(X_train_scaled)
    test_preds = model.predict(X_test_scaled)

    # 4. Calculate Accuracies
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)

    print(f"\n{'='*40}")
    print(f" DISEASE: {disease.replace('_', ' ')}")
    print(f"{'='*40}")
    print(f"Training Accuracy: {train_acc:.3f}")
    print(f"Testing Accuracy:  {test_acc:.3f}")
    
    # 5. Check the gap
    gap = train_acc - test_acc
    if gap > 0.05:
        print(f"⚠ WARNING: High overfitting detected (Gap: {gap:.3f})")
    else:
        print(f" Model is generalizing well (Gap: {gap:.3f})")

    # 6. Detailed Performance Report
    print("\nDetailed Classification Report (Testing Data):")
    print(classification_report(y_test, test_preds))

    models[disease] = model

    # Save each model
    models[disease] = model
    joblib.dump(model, f"{disease}_model.pkl")

print("\nModels and Scaler saved successfully.")


import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# --- 1. Load and Prepare Data ---
file_path = "health_nutrition_disease_dataset_12000.xlsx"
df = pd.read_excel(file_path)

# Encoding
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})
disease_columns = ["Diabetes_Risk", "Hypertension_Risk", "Heart_Disease_Risk", "Obesity_Risk", "Anemia_Risk", "Kidney_Disease_Risk"]
for col in disease_columns:
    df[col] = df[col].map({"High": 1, "Low": 0})

features = ["Age", "Gender", "BMI", "Daily_Calories_kcal", "Carbohydrates_g", "Protein_g", "Total_Fat_g", 
            "Saturated_Fat_g", "Trans_Fat_g", "Total_Sugar_g", "Added_Sugar_g", "Fiber_g", "Sodium_mg", 
            "Potassium_mg", "Calcium_mg", "Iron_mg", "Vitamin_D_IU", "Vitamin_B12_mcg", "Physical_Activity_min", "Water_Intake_L"]

X = df[features]
X_train_raw, X_test_raw = train_test_split(X, test_size=0.2, random_state=42)

# Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
joblib.dump(scaler, "scaler.pkl")

# --- 2. Train Models with Anti-Overfitting Settings ---
models = {}
print("--- MODEL TRAINING PERFORMANCE ---")

for disease in disease_columns:
    y = df[disease]
    y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)

    model = XGBClassifier(
    n_estimators=50,
    max_depth=2,
    scale_pos_weight=5,  # Forces the model to pay 5x more attention to High Risk cases
    learning_rate=0.01,    # Very slow learning for higher precision
    reg_alpha=5,            # L1 regularization to ignore "noisy" features
    reg_lambda=5,           # L2 regularization to smooth out predictions
    min_child_weight=10,    # Requires more patients to agree before making a rule
    eval_metric="logloss"
)
    model.fit(X_train_scaled, y_train)
    
    # Check for Overfitting
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    
    print(f"\nTarget: {disease}")
    print(f"Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f} (Gap: {train_acc-test_acc:.3f})")
    
    models[disease] = model
    joblib.dump(model, f"{disease}_model.pkl")

# --- 3. Run Predictions on Multiple Patients ---
def calculate_bmi(weight_kg, height_cm):
    return round(weight_kg / ((height_cm / 100) ** 2), 1)

# List of Dummy Patients
test_patients = [
    {"Name": "Patient 1 ()", "Age": 45, "Gender": 1, "Weight_kg": 70, "Height_cm": 154, "Daily_Calories_kcal": 2800, "Carbohydrates_g": 350, "Protein_g": 90, "Total_Fat_g": 130, "Saturated_Fat_g": 45, "Trans_Fat_g": 2, "Total_Sugar_g": 120, "Added_Sugar_g": 90, "Fiber_g": 15, "Sodium_mg": 3500, "Potassium_mg": 2000, "Calcium_mg": 500, "Iron_mg": 7, "Vitamin_D_IU": 2190, "Vitamin_B12_mcg": 1.8, "Physical_Activity_min": 20, "Water_Intake_L": 1.2},
    {"Name": "Patient 2 (High Risk)", "Age": 62, "Gender": 0, "Weight_kg": 95, "Height_cm": 170, "Daily_Calories_kcal": 3200, "Carbohydrates_g": 450, "Protein_g": 70, "Total_Fat_g": 140, "Saturated_Fat_g": 60, "Trans_Fat_g": 5, "Total_Sugar_g": 180, "Added_Sugar_g": 130, "Fiber_g": 10, "Sodium_mg": 4800, "Potassium_mg": 1500, "Calcium_mg": 400, "Iron_mg": 6, "Vitamin_D_IU": 800, "Vitamin_B12_mcg": 1.2, "Physical_Activity_min": 5, "Water_Intake_L": 0.8},
    {"Name": "Patient 3 (Healthy)", "Age": 28, "Gender": 1, "Weight_kg": 60, "Height_cm": 165, "Daily_Calories_kcal": 2100, "Carbohydrates_g": 250, "Protein_g": 120, "Total_Fat_g": 70, "Saturated_Fat_g": 15, "Trans_Fat_g": 0, "Total_Sugar_g": 40, "Added_Sugar_g": 10, "Fiber_g": 35, "Sodium_mg": 1800, "Potassium_mg": 3500, "Calcium_mg": 1200, "Iron_mg": 18, "Vitamin_D_IU": 4000, "Vitamin_B12_mcg": 4.5, "Physical_Activity_min": 90, "Water_Intake_L": 3.0}
]

print("\n" + "="*50 + "\nFINAL PREDICTIONS\n" + "="*50)

for p in test_patients:
    p["BMI"] = calculate_bmi(p["Weight_kg"], p["Height_cm"])
    p_df = pd.DataFrame([{k: p[k] for k in features}])
    p_scaled = scaler.transform(p_df)
    
    print(f"\n>>> {p['Name']}")
    for disease in disease_columns:
        # Using predict_proba to see how "sure" the model is
        prob = models[disease].predict_proba(p_scaled)[0][1] 
        status = "️ HIGH RISK" if prob > 0.30 else " LOW RISK"
        print(f"{disease.replace('_', ' '):<20}: {status} ({prob*100:.1f}%)")

Dataset loaded: (12000, 26)

--- Training Results ---

 DISEASE: Diabetes Risk
Training Accuracy: 0.999
Testing Accuracy:  0.999
 Model is generalizing well (Gap: 0.000)

Detailed Classification Report (Testing Data):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1746
           1       1.00      1.00      1.00       654

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg       1.00      1.00      1.00      2400


 DISEASE: Hypertension Risk
Training Accuracy: 0.999
Testing Accuracy:  1.000
 Model is generalizing well (Gap: -0.001)

Detailed Classification Report (Testing Data):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1845
           1       1.00      1.00      1.00       555

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg  